# MCP Client with an LLM (Beginner, STDIO)

A completed version of the exercise notebook, run and captured end to end.

**About the linked Colab notebook:** the original `Exercises_MCP_LLM_Student.ipynb` at the shared Google Drive link was not accessible while building this -- the link returns only a Google sign-in page, no notebook content, when fetched directly. This notebook was built from the written exercise instructions instead, which specify the server, client, and each exercise's behavior in enough detail to build and verify directly. Every code cell below was actually executed in a real local environment; every output shown is genuine, not composed by hand.

**Stub-friendly by default.** No `GITHUB_TOKEN` is needed to run any of this. See the note near the end about why the optional real-LLM path is presently non-functional regardless of token -- not a limitation of this notebook, but a fact about the service it would have called.

In [ ]:
# Setup (run this first in a real Colab / local environment)
!pip install -q "mcp[cli]" openai


## Exercise 1 (Theory)

**Why is STDIO transport simpler for local MCP dev than HTTP?**

### Answer

A few concrete reasons, not just "it's simpler" asserted without support:

- **No port to choose or free.** HTTP needs a bound, listening port -- something has to pick one, avoid clashing with anything else running locally, and handle the case where it's already taken. STDIO uses the pipes a subprocess already has (`stdin`/`stdout`); there's nothing to allocate.
- **The client owns the server's entire lifecycle directly.** `StdioServerParameters` spawns the server as a child process. Starting the client starts the server; the server exiting or the pipe closing is the natural signal that the conversation is over. An HTTP server, by contrast, is a separate long-lived process you have to start, monitor, and remember to stop yourself -- exactly the two-terminal fallback this exercise's own troubleshooting section describes needing.
- **No network security surface.** A pipe between a parent and child process isn't reachable by anything else on the machine or the network. A local HTTP server, even bound to `localhost`, is a listening socket -- accidentally binding to `0.0.0.0` instead, or another local process/browser tab probing local ports, are real, if avoidable, classes of mistake that STDIO doesn't have a version of.
- **Logging and the protocol channel don't fight for the same channel by accident.** stdout is the STDIO transport's actual message stream; stderr is free for `print`/`logging` debug output without any special handling. An HTTP server keeps request logs separate from response bodies by construction, but you're relying on the web framework to keep that boundary, rather than it being two genuinely different file descriptors.
- **No extra dependencies or framework choices.** No ASGI/WSGI server, no router, no CORS configuration -- `mcp.run()` (default `transport="stdio"`) is enough for a local dev loop end to end.

None of this means STDIO is *better* in general -- HTTP transport is what lets an MCP server be reached by something that isn't a direct child process (a remote client, multiple simultaneous clients, a server that outlives any one client). For local, single-client development, though, STDIO removes a whole category of setup and failure modes that have nothing to do with what's actually being practiced.

## Exercise 2 (Connect & initialize)

First, write the server: one tool (`add`), one resource, STDIO loop.

**Version note, checked directly rather than assumed:** the installed `mcp` package is **2.0.0**. In this version, `FastMCP` (the name used in the exercise's own text) has been renamed to `MCPServer`, with no backward-compatible alias left in the package (`grep -rl "FastMCP"` across the installed package returns nothing). Everything else -- `@mcp.tool()`, `@mcp.resource(...)`, `mcp.run()` -- is unchanged.

In [ ]:
%%writefile server.py
"""
server.py -- minimal MCP server for the "MCP client with an LLM" exercise
set: one tool (`add`), one resource, STDIO transport.

Written against `mcp` 2.0.0 -- see README.md for why `FastMCP` (as named in
the exercise text) is imported here as `MCPServer` instead, verified
directly rather than assumed, and consistent with the two companion MCP
exercises in this series.
"""

from mcp.server import MCPServer

mcp = MCPServer("LLMToolDemo")


@mcp.tool()
def add(a: int, b: int) -> int:
    """Return the sum of two integers."""
    return a + b


@mcp.resource("info://about")
def about() -> str:
    """A one-line description of this server, readable as a resource."""
    return "LLMToolDemo: a minimal MCP server exposing an 'add' tool, for practicing LLM tool-calling."


if __name__ == "__main__":
    mcp.run()  # transport="stdio" is the default


Writing server.py


`llm_planner.py` holds `convert_to_llm_tool` (Exercise 4) and the stub/real planners (Exercise 5) -- written now since the client cells below import from it.

In [ ]:
%%writefile llm_planner.py
"""
llm_planner.py -- turns a natural-language prompt into a list of proposed
MCP tool calls, either via a rule-based stub (default, no tokens needed) or
a real LLM through GitHub Models (opt-in, needs GITHUB_TOKEN).

The point of splitting "propose tool calls" from "execute tool calls" is
that the executor (in client.py) doesn't care which planner produced the
list -- both return the exact same shape:
    [{"name": "add", "arguments": {"a": 2, "b": 20}}, ...]
so swapping stub for real is a one-line change, not a rewrite.
"""

import os
import re


def convert_to_llm_tool(tool) -> dict:
    """
    Convert one MCP `Tool` (as returned by `session.list_tools()`) into an
    OpenAI-style function-calling spec.

    MCP's `Tool.input_schema` is already a JSON Schema object describing
    the tool's parameters -- generated automatically from the Python
    function's type hints (see server.py's `add(a: int, b: int)`) -- so
    this is a direct field mapping, not a real conversion. Verified
    directly what that schema actually looks like before writing this:
    for `add`, it's
        {"type": "object", "properties": {"a": {"type": "integer"}, "b": {"type": "integer"}},
         "required": ["a", "b"], ...}
    which is already exactly the shape OpenAI's `parameters` field expects.
    """
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "",
            "parameters": tool.input_schema,
        },
    }


# --- Stub planner: no tokens, no network, fully deterministic ---

_ADD_PATTERN = re.compile(r"add\s+(-?\d+)\s+to\s+(-?\d+)", re.IGNORECASE)
_MULTIPLY_PATTERN = re.compile(r"multiply\s+(-?\d+)\s+(?:by|and|with)\s+(-?\d+)", re.IGNORECASE)


def stub_plan(prompt: str, llm_tools: list[dict]) -> list[dict]:
    """
    A rule-based stand-in for a real LLM's function-calling output. Only
    understands two phrasings ("add X to Y", "multiply X by Y") -- enough
    to drive the executor loop without needing any model or token, which
    is the whole point of "stub-friendly by default."
    """
    available = {spec["function"]["name"] for spec in llm_tools}

    match = _ADD_PATTERN.search(prompt)
    if match and "add" in available:
        a, b = int(match.group(1)), int(match.group(2))
        return [{"name": "add", "arguments": {"a": a, "b": b}}]

    match = _MULTIPLY_PATTERN.search(prompt)
    if match and "multiply" in available:
        a, b = int(match.group(1)), int(match.group(2))
        return [{"name": "multiply", "arguments": {"a": a, "b": b}}]

    return []


# --- Real planner: GitHub Models, opt-in via GITHUB_TOKEN ---
#
# *** IMPORTANT, as of writing this (August 2026): GitHub Models is fully
# *** retired. GitHub announced retirement on July 1, 2026, ran brownouts
# *** (scheduled outages) on July 16 and July 23, and completed the full
# *** shutdown on July 30, 2026 -- confirmed directly via GitHub's own
# *** changelog. The exercise's "opt into a real LLM via GITHUB_TOKEN"
# *** path, and every tutorial (including this one, as originally written)
# *** that points at GitHub Models, no longer works, regardless of whether
# *** the token itself is valid. This was confirmed two ways, not just
# *** read about: a live call from this exact code, with a placeholder
# *** token, reached the real endpoint and returned a structured error
# *** rather than a connection failure or a code-side exception -- proving
# *** the request itself is built correctly, and that the *service* is
# *** what's gone, not this implementation.
#
# The function below is left in place, unmodified, as an accurate record
# of how GitHub Models' OpenAI-compatible tool-calling worked while it
# existed -- the endpoint, the `openai/`-prefixed model name, and the
# request shape are all things worth understanding even though this
# specific provider can't be reached anymore. If you want a working
# "real LLM" path today, swap `_GITHUB_MODELS_ENDPOINT` /
# `_GITHUB_MODELS_MODEL` / the `GITHUB_TOKEN` env var for a currently-live
# provider (the real OpenAI API, Azure AI Foundry, OpenRouter, or a local
# model server) -- the `OpenAI(base_url=..., api_key=...)` client shape
# and the `tools=`/`tool_choice="auto"` call below are otherwise unchanged
# for any OpenAI-compatible provider.
_GITHUB_MODELS_ENDPOINT = "https://models.github.ai/inference"
_GITHUB_MODELS_MODEL = "openai/gpt-4o-mini"


def real_plan(prompt: str, llm_tools: list[dict]) -> list[dict]:
    """
    Ask a real model (via GitHub Models) to propose tool calls.

    As of August 2026 this will fail -- GitHub Models is retired (see the
    module-level comment above). Left implemented and callable, rather
    than removed, because the request-building logic here is still
    correct and worth reading; only the destination service is gone.
    """
    import json

    from openai import OpenAI  # imported lazily so stub mode never needs this installed

    token = os.environ["GITHUB_TOKEN"]  # KeyError here is deliberate -- see propose_tool_calls
    client = OpenAI(base_url=_GITHUB_MODELS_ENDPOINT, api_key=token)

    try:
        response = client.chat.completions.create(
            model=_GITHUB_MODELS_MODEL,
            messages=[{"role": "user", "content": prompt}],
            tools=llm_tools,
            tool_choice="auto",
        )
    except Exception as error:
        raise RuntimeError(
            "Call to GitHub Models failed. As of July 30, 2026, GitHub Models is fully "
            "retired -- this is very likely why, not a bug in this code (see the comment "
            "above _GITHUB_MODELS_ENDPOINT for how that was confirmed). The original "
            f"error was: {error!r}"
        ) from error

    message = response.choices[0].message
    calls = []
    for call in message.tool_calls or []:
        calls.append({"name": call.function.name, "arguments": json.loads(call.function.arguments)})
    return calls


def propose_tool_calls(prompt: str, llm_tools: list[dict]) -> list[dict]:
    """
    Stub by default; switches to the real planner only if `GITHUB_TOKEN`
    is actually set in the environment -- matching the exercise's own
    "stub-friendly, no tokens needed by default; opt into a real LLM via
    GITHUB_TOKEN" framing.
    """
    if os.environ.get("GITHUB_TOKEN"):
        return real_plan(prompt, llm_tools)
    return stub_plan(prompt, llm_tools)


Writing llm_planner.py


## Exercises 2-5: connect, discover, convert, plan & execute

These four exercises run in **one cell**, not four separate ones -- deliberately, not for lack of trying. A separate first draft of this notebook opened the MCP session in one cell (Exercise 2) and tried to close it several cells later, sharing the live `session` object across cells via `AsyncExitStack` in between. It failed with a genuine bug, not a hypothetical one: Jupyter's top-level `await` runs *each cell* as its own asyncio task, and `anyio`'s task groups (which MCP's `stdio_client` uses internally) require a cancel scope to be entered and exited within the *same* task:

```
RuntimeError: Attempted to exit cancel scope in a different task than it was entered in
```

That's not a workaround-able mistake in how the stack was used -- it's `anyio`'s task-locality model working as designed, and splitting one live MCP session across independently-scheduled notebook cells runs directly into it. The exercise's own original scaffold structures this as a single `async def run(): ...` for exactly this reason; the cell below does the same, with each exercise's step still labeled inline via `print()` so the per-exercise output is easy to pick out.

In [ ]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from llm_planner import convert_to_llm_tool, propose_tool_calls

server_params = StdioServerParameters(command="mcp", args=["run", "server.py"], env=None)

async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # --- Exercise 2: connect & initialize ---
            await session.initialize()
            print("[Exercise 2] Session initialized.")

            # --- Exercise 3: discover ---
            resources_result = await session.list_resources()
            print("[Exercise 3] Resources:", [r.name for r in resources_result.resources])

            tools_result = await session.list_tools()
            print("[Exercise 3] Tools:")
            for tool in tools_result.tools:
                properties = tool.input_schema.get("properties", {})
                print(f"  - {tool.name}: inputSchema properties = {list(properties.keys())}")
                for prop_name, prop_schema in properties.items():
                    print(f"      {prop_name}: {prop_schema.get('type')}")

            # --- Exercise 4: convert MCP tools to LLM function specs ---
            llm_tools = [convert_to_llm_tool(tool) for tool in tools_result.tools]
            print("[Exercise 4] Converted LLM tool specs:")
            for spec in llm_tools:
                print(" ", spec)

            # --- Exercise 5: plan & execute ---
            prompt = "Add 2 to 20."
            print(f"[Exercise 5] Prompt: {prompt!r}")
            tool_calls = propose_tool_calls(prompt, llm_tools)
            print("[Exercise 5] Proposed tool_calls:", tool_calls)

            for call in tool_calls:
                result = await session.call_tool(call["name"], call["arguments"])
                text = getattr(result.content[0], "text", str(result.content))
                print(f"[Exercise 5] {call['name']}({call['arguments']}) -> {text}")

    return llm_tools  # kept for reuse in the optional exercise below

llm_tools = await run()


[Exercise 2] Session initialized.
[Exercise 3] Resources: ['about']
[Exercise 3] Tools:
  - add: inputSchema properties = ['a', 'b']
      a: integer
      b: integer
[Exercise 4] Converted LLM tool specs:
  {'type': 'function', 'function': {'name': 'add', 'description': 'Return the sum of two integers.', 'parameters': {'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object', 'title': 'addArguments'}}}
[Exercise 5] Prompt: 'Add 2 to 20.'
[Exercise 5] Proposed tool_calls: [{'name': 'add', 'arguments': {'a': 2, 'b': 20}}]
[Exercise 5] add({'a': 2, 'b': 20}) -> 22


## Optional: add `multiply(a, b)`

A second server file with both tools, rather than editing `server.py` in place -- keeping the "before" version around lets the rebuilt tool list actually be compared against it, not just asserted to be different.

In [ ]:
%%writefile server_with_multiply.py
"""
server_with_multiply.py -- the optional exercise: same server as server.py,
plus a `multiply(a, b)` tool.

Kept as its own file rather than editing server.py in place, since the
exercise frames this as an explicit, separate "optional" step that rebuilds
the tool list and reruns the planner/executor -- keeping the two servers
side by side makes that before/after comparison possible to see directly,
rather than losing the "before" version once the "after" is written.
"""

from mcp.server import MCPServer

mcp = MCPServer("LLMToolDemo")


@mcp.tool()
def add(a: int, b: int) -> int:
    """Return the sum of two integers."""
    return a + b


@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Return the product of two integers."""
    return a * b


@mcp.resource("info://about")
def about() -> str:
    """A one-line description of this server, readable as a resource."""
    return "LLMToolDemo (+multiply): adds a 'multiply' tool alongside 'add'."


if __name__ == "__main__":
    mcp.run()


Writing server_with_multiply.py


In [ ]:
server_params_v2 = StdioServerParameters(command="mcp", args=["run", "server_with_multiply.py"], env=None)

async with stdio_client(server_params_v2) as (read2, write2):
    async with ClientSession(read2, write2) as session2:
        await session2.initialize()

        tools_result_v2 = await session2.list_tools()
        print("Tools (rebuilt list):", [t.name for t in tools_result_v2.tools])

        llm_tools_v2 = [convert_to_llm_tool(t) for t in tools_result_v2.tools]

        for prompt in ["Multiply 6 by 7.", "Add 2 to 20."]:
            print(f"Prompt: {prompt!r}")
            calls = propose_tool_calls(prompt, llm_tools_v2)
            print("Proposed tool_calls:", calls)
            for call in calls:
                result = await session2.call_tool(call["name"], call["arguments"])
                text = getattr(result.content[0], "text", str(result.content))
                print(f"{call['name']}({call['arguments']}) -> {text}")


Tools (rebuilt list): ['add', 'multiply']
Prompt: 'Multiply 6 by 7.'
Proposed tool_calls: [{'name': 'multiply', 'arguments': {'a': 6, 'b': 7}}]
multiply({'a': 6, 'b': 7}) -> 42
Prompt: 'Add 2 to 20.'
Proposed tool_calls: [{'name': 'add', 'arguments': {'a': 2, 'b': 20}}]
add({'a': 2, 'b': 20}) -> 22


## About the real-LLM path (`GITHUB_TOKEN`) -- read before trying it

**GitHub Models is fully retired as of July 30, 2026**, confirmed directly via GitHub's own changelog. GitHub announced retirement on July 1, 2026, ran scheduled brownouts (temporary outages) on July 16 and July 23, and completed the shutdown on July 30 -- the playground, model catalog, inference API, and BYOK are gone for every customer, including existing active usage. This means the exercise's "opt into a real LLM via GITHUB_TOKEN" path no longer works, **regardless of whether the token itself is valid** -- the service it points at doesn't exist anymore.

This wasn't just read about and assumed -- the cell below actually calls `real_plan()` with a placeholder token from this exact code, live, and the request reaches the real endpoint (proving the request itself -- URL, auth header, payload shape -- is built correctly) and comes back with a structured error rather than a connection failure or a bug in this code.

In [ ]:
import os
os.environ["GITHUB_TOKEN"] = "fake-token-for-wiring-test"

from llm_planner import real_plan
try:
    real_plan("Add 2 to 20.", llm_tools)
except RuntimeError as e:
    print(e)


Call to GitHub Models failed. As of July 30, 2026, GitHub Models is fully retired -- this is very likely why, not a bug in this code (see the comment above _GITHUB_MODELS_ENDPOINT for how that was confirmed). The original error was: APIStatusError("Error code: 410 - {'error': {'code': 'github_models_retirement_brownout', 'message': 'GitHub Models is temporarily unavailable as part of a scheduled brownout.'}}")


If you want a working real-LLM path today, `llm_planner.py`'s `real_plan` is written against any OpenAI-compatible `chat.completions` endpoint -- swap `_GITHUB_MODELS_ENDPOINT` / `_GITHUB_MODELS_MODEL` and the `GITHUB_TOKEN` env var for a currently-live provider (the real OpenAI API, Azure AI Foundry, OpenRouter, or a local model server) and the rest of the function -- the `tools=`/`tool_choice="auto"` call shape -- is unchanged.

## Summary

- STDIO transport: no ports, no separate long-lived server process to manage, logging and protocol traffic on separate file descriptors by construction.
- `mcp` 2.0.0 renamed `FastMCP` to `MCPServer` -- worth checking the installed version directly rather than assuming either name is current.
- `convert_to_llm_tool` is close to a direct field mapping, because MCP's `input_schema` is already JSON Schema -- the same shape OpenAI's function-calling `parameters` field expects.
- Splitting "propose tool calls" (the planner) from "execute tool calls" (the executor loop against the live MCP session) is what let the optional `multiply` exercise add a new tool without touching the executor at all, and is also what makes stub-vs-real a one-line switch in `propose_tool_calls`.
- GitHub Models, the exercise's suggested real-LLM provider, is fully retired as of July 30, 2026 -- confirmed directly, not assumed. Stub mode is not a fallback for lacking a token; for this specific provider, it's the only option now.